# Подготовка данных для обучения моделей

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы: 
* https://scikit-learn.org/stable/modules/compose.html#pipeline-chaining-estimators
* https://pytorch.org/docs/stable/data.html 
* https://pytorch.org/tutorials/beginner/data_loading_tutorial.html
* Deep Learning with PyTorch (2020) Авторы: Eli Stevens, Luca Antiga, Thomas Viehmann


## Задачи для совместного разбора

1. Создайте синтетический датасет для задачи регрессии и представьте его в виде `torch.utils.data.Dataset`

## Задачи для самостоятельного решения

<p class="task" id="1"></p>

1\. Считайте файл `bank-full.csv` ([источник](https://www.kaggle.com/datasets/hariharanpavan/bank-marketing-dataset-analysis-classification)) в виде `pd.DataFrame`.

Опишите класс `BankDatasetBase`. Решение должно удовлетворять следующим критериям:

* класс наследуется от `torch.utils.data.Dataset`;
* при создании объекта в конструктор передается набор данных в виде `pd.DataFrame`;
* объекты класса имеют поля `X` и `y` с признаками и метками соответственно;
* класс реализует интерфейс последовательностей (`__getitem__` + `__len__`);
* `obj[i]` возвращает кортеж, содержащий `i`-ую строку из `obj.X` (серию) и `i`-ую строку из `obj.y` (строку).
    
Создайте объект класса `BankDatasetBase` и продемонстрируйте работоспособность.

- [ ] Проверено на семинаре

In [1]:
import pandas as pd
import torch
import torch.nn as nn

In [2]:
class BankDatasetBase(torch.utils.data.Dataset):
    def __init__(self, data: pd.DataFrame) -> None:
        if not isinstance(data, pd.DataFrame):
            raise TypeError()
        super().__init__()
        self.data = data
        self.X = data.drop(columns=['y'])   # все колонки кроме y
        self.y = data['y']

    def __getitem__(self, idx: int) -> tuple:
        return self.X.iloc[idx], self.y.iloc[idx]
    
    def __len__(self) -> int:
        return len(self.X)

In [5]:
df = pd.read_csv('bank-full.csv')
dataset = BankDatasetBase(df)

print(f"Длина датасета: {len(dataset)}")
print(f"Тип dataset.X: {type(dataset.X)}")
print(f"Тип dataset.y: {type(dataset.y)}")

example = dataset[0]
print(type(example[0]), example[0].shape)
print(type(example[1]), example[1])

Длина датасета: 45211
Тип dataset.X: <class 'pandas.DataFrame'>
Тип dataset.y: <class 'pandas.Series'>
<class 'pandas.Series'> (16,)
<class 'str'> no


<p class="task" id="2"></p>

2\. Опишите класс `BankDataset`. Решение должно удовлетворять всем критериям из предыдущего задания, а также:
* при создании объекта в конструктор может быть передан необязательные аргументы `transform` и `target_transform`;
* если аргумент `transform` был передан, то при получении `i`-го элемента, нужно вызвать `transform(x)` и вернуть полученный результат.
* если аргумент `target_transform` был передан, то при получении `i`-го элемента, нужно вызвать `target_transform(y)` и вернуть полученный результат.

Создайте объект класса `BankDataset` и продемонстрируйте работоспособность (без передачи `target_transform` и `transform`).

- [ ] Проверено на семинаре

In [6]:
from typing import Callable


class BankDataset(BankDatasetBase):
    def __init__(
            self, 
            data: pd.DataFrame, 
            transform: Callable | None = None,
            target_transform: Callable | None = None
    ) -> None:
        super().__init__(data)
        self.transform = transform
        self.target_transform = target_transform

    def __getitem__(self, idx: int) -> tuple:
        x, y = super().__getitem__(idx)

        if self.transform is not None:
            x = self.transform(x)

        if self.target_transform is not None:
            y = self.target_transform(y)

        return x, y

    def __len__(self) -> int:
        return super().__len__()

In [7]:
df.tail(20)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
45191,75,retired,divorced,tertiary,no,3810,yes,no,cellular,16,nov,262,1,183,1,failure,yes
45192,29,management,single,tertiary,no,765,no,no,cellular,16,nov,238,1,-1,0,unknown,yes
45193,28,self-employed,single,tertiary,no,159,no,no,cellular,16,nov,449,2,33,4,success,yes
45194,59,management,married,tertiary,no,138,yes,yes,cellular,16,nov,162,2,187,5,failure,no
45195,68,retired,married,secondary,no,1146,no,no,cellular,16,nov,212,1,187,6,success,yes
45196,25,student,single,secondary,no,358,no,no,cellular,16,nov,330,1,-1,0,unknown,yes
45197,36,management,single,secondary,no,1511,yes,no,cellular,16,nov,270,1,-1,0,unknown,yes
45198,37,management,married,tertiary,no,1428,no,no,cellular,16,nov,333,2,-1,0,unknown,no
45199,34,blue-collar,single,secondary,no,1475,yes,no,cellular,16,nov,1166,3,530,12,other,no
45200,38,technician,married,secondary,no,557,yes,no,cellular,16,nov,1556,4,-1,0,unknown,yes


In [8]:
dataset = BankDataset(df)

print(f"Длина датасета: {len(dataset):,}")

sample = dataset[0]
print("\nПример dataset[0]:")
print(type(sample[0]))
print(type(sample[1]))
print(sample[1])

Длина датасета: 45,211

Пример dataset[0]:
<class 'pandas.Series'>
<class 'str'>
no


<p class="task" id="3"></p>

3\. Опишите класс `OrdinalEncoderTransform`. Решение должно удовлетворять следующим критериям:

* при создании объекта в конструктор передаются названия нечисловых столбцов в датасете
* класс реализует интерфейс `Callable` (`__call__`); метод `__call__` имеет один параметр (признаки) и возвращает набор признаков, в котором нечисловые характеристики закодированы целыми числами;
* состояние объекта (индексы для кодирования) обновляется в момент очередного вызова `__call__` (т.е. все данные сразу никогда не передаются никакому методу объекта).
    
Продемонстрируйте работоспособность, создав объект `BankDataset` и передав при создании объект класса `OrdinalEncoderTransform`.

- [ ] Проверено на семинаре

In [9]:
class Transform:
    pass

In [10]:
class OrdinalEncoderTransform(Transform):
    def __init__(self, category_columns: list[str]) -> None:
        self.category_columns = category_columns
        # Состояние объекта
        self.mappings: dict[str, dict] = {col: {} for col in category_columns}
        self.next_index: dict[str, int] = {col: 0 for col in category_columns}

    def __call__(self, x: pd.Series) -> pd.Series:
        x = x.copy()
        for col in self.category_columns:
            if col in x:
                val = x[col]
                if val not in self.mappings[col]:
                    self.mappings[col][val] = self.next_index[col]
                    self.next_index[col] += 1
                
                x[col] = self.mappings[col][val]

        return x

In [11]:
df = pd.read_csv("bank-full.csv")

category_columns = [
    'job', 'marital', 'education', 'default',
    'housing', 'loan', 'contact', 'month', 'poutcome'
]

ordinal_transform = OrdinalEncoderTransform(category_columns)
dataset = BankDataset(df, transform=ordinal_transform)

print(f"Длина датасета: {len(dataset):,}")

print("\nИсходная строка 0 (до кодирования):")
print(df.iloc[0][category_columns])
print("\nПосле применения transform (dataset[0]):")
x_encoded, y = dataset[0]
print(x_encoded[category_columns])

for i in range(1, 45211):
    _ = dataset[i]

print("\nПосле 20 строк:")
print(f"job - {len(ordinal_transform.mappings['job'])} категорий")
print(f"month - {len(ordinal_transform.mappings['month'])} категорий")
print(f"poutcome - {len(ordinal_transform.mappings['poutcome'])} категорий")

Длина датасета: 45,211

Исходная строка 0 (до кодирования):
job          management
marital         married
education      tertiary
default              no
housing             yes
loan                 no
contact         unknown
month               may
poutcome        unknown
Name: 0, dtype: object

После применения transform (dataset[0]):
job          0
marital      0
education    0
default      0
housing      0
loan         0
contact      0
month        0
poutcome     0
Name: 0, dtype: object

После 20 строк:
job - 12 категорий
month - 12 категорий
poutcome - 4 категорий


<p class="task" id="4"></p>

4\. Опишите класс `LabelEncoderTransform`. Решение должно удовлетворять следующим критериям:

* класс реализует интерфейс `Callable` (`__call__`); метод `__call__` имеет один параметр (строку) и возвращает целое число, соответствующее этой строке;
* состояние объекта (индексы для кодирования) обновляется в момент очередного вызова `__call__` (т.е. все данные сразу никогда не передаются никакому методу объекта).
    
Продемонстрируйте работоспособность, создав объект `BankDataset` и передав при создании объекта в качестве аргумента `target_transform` объект класса `LabelEncoderTransform`.

- [ ] Проверено на семинаре

In [12]:
class LabelEncoderTransform(Transform):
    def __init__(self) -> None:
        self.mapping: dict[str, int] = {}
        self.next_index: int = 0

    def __call__(self, x: str) -> int:
        if x not in self.mapping:
            self.mapping[x] = self.next_index
            self.next_index += 1
        return self.mapping[x]

In [13]:
label_encoder = LabelEncoderTransform()

dataset = BankDataset(df, target_transform=label_encoder)

print(f"Длина датасета: {len(dataset):,}")

for i in range(10):
    original_y = df.iloc[i]['y']
    encoded_y = dataset[i][1]
    print(f"   dataset[{i}][1] = {encoded_y}  (оригинал: '{original_y}')")

print(label_encoder.mapping)
print(label_encoder.next_index)


_ = [dataset[i] for i in range(100)]
label_encoder.mapping, label_encoder.next_index

Длина датасета: 45,211
   dataset[0][1] = 0  (оригинал: 'no')
   dataset[1][1] = 0  (оригинал: 'no')
   dataset[2][1] = 0  (оригинал: 'no')
   dataset[3][1] = 0  (оригинал: 'no')
   dataset[4][1] = 0  (оригинал: 'no')
   dataset[5][1] = 0  (оригинал: 'no')
   dataset[6][1] = 0  (оригинал: 'no')
   dataset[7][1] = 0  (оригинал: 'no')
   dataset[8][1] = 0  (оригинал: 'no')
   dataset[9][1] = 0  (оригинал: 'no')
{'no': 0}
1


({'no': 0, 'yes': 1}, 2)

<p class="task" id="5"></p>

5\. Опишите класс `ToTensor`.  Решение должно удовлетворять следующим критериям:
* класс реализует интерфейс `Callable` (`__call__`); метод `__call__` принимает на вход серию или фрейм и возвращает тензор.

Опишите класс `Compose`.  Решение должно удовлетворять следующим критериям:
* при создании объекта в конструктор передается список объектов `transforms`, каждый из которых имеет метод `__call__(x, y)`;
* класс реализует интерфейс `Callable` (`__call__`); метод `__call__` принимает имеет параметра (признаки и класс в числовом виде) и и возвращает кортеж, полученный путем последовательного вызова объектов из `transforms`.

Продемонстрируйте работоспособность, создав объект `BankDataset` и передав при создании преобразования `Compose` список из объектов LabelEncoderTransform и ToTensor.

- [ ] Проверено на семинаре

In [26]:
import torch as th
from typing import Any

class ToTensor(Transform):
    def __call__(self, X: pd.Series | int) -> th.Tensor:
        if isinstance(X, pd.Series):
            return th.tensor(X.values.astype(float), dtype=th.float32)
        else:
            return th.tensor(int(X), dtype=th.long)


class Compose(Transform):
    def __init__(self, transforms: list[Transform]) -> None:
        self.transforms = transforms

    def __call__(self, X: Any) -> Any:
        for transform in self.transforms:
            X = transform(X)
        return X

In [22]:
target_transform = Compose([
    LabelEncoderTransform(),
    ToTensor()
])

dataset = BankDataset(df, target_transform=target_transform)

print(f"Длина датасета: {len(dataset):,}")


print("\nПримеры dataset[i] (y теперь тензор):")
for i in range(5):
    x, y_tensor = dataset[i]
    print(f"   dataset[{i}][1] = {y_tensor}  (тип: {type(y_tensor)}, dtype: {y_tensor.dtype})")

print("\nСостояние LabelEncoder внутри Compose:")
label_encoder = target_transform.transforms[0]   # первый трансформер в списке
print(label_encoder.mapping)
print(label_encoder.next_index)

Длина датасета: 45,211

Примеры dataset[i] (y теперь тензор):
   dataset[0][1] = 0  (тип: <class 'torch.Tensor'>, dtype: torch.int64)
   dataset[1][1] = 0  (тип: <class 'torch.Tensor'>, dtype: torch.int64)
   dataset[2][1] = 0  (тип: <class 'torch.Tensor'>, dtype: torch.int64)
   dataset[3][1] = 0  (тип: <class 'torch.Tensor'>, dtype: torch.int64)
   dataset[4][1] = 0  (тип: <class 'torch.Tensor'>, dtype: torch.int64)

Состояние LabelEncoder внутри Compose:
{'no': 0}
1


<p class="task" id="6"></p>

6\. Разделите датасет из предыдущего задания на обучающую и тестовую выборку в соотношении 75% на 25%. Создайте объект `DataLoader` для получения пакетов размера 64, полученных из перемешанного обучающего датасета. Кастомизируйте `DataLoader` таким образом, чтобы пакет признаков был представлен в виде трехмерного тензора размера 64x2x8 (разделите 16 признаков на два тензора по 8). Получите один пакет и выведите на экран размерность тензоров пакета. 

- [ ] Проверено на семинаре

In [28]:
data = pd.read_csv("bank-full.csv")

dataset = BankDataset(
    data,
    transform=Compose([
        OrdinalEncoderTransform([
            "job","marital","education","default",
            "housing","loan","contact","month","poutcome"
        ]),
        ToTensor()
    ]),
    target_transform=Compose([
        LabelEncoderTransform(),
        ToTensor()
    ])
)

In [29]:
train_size = int(0.75 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])


def custom_collate(batch):
    features, targets = zip(*batch)

    features = torch.stack(features)   # (64, 16)
    targets = torch.stack(targets)

    features = features.view(-1, 2, 8) # (64, 2, 8)

    return features, targets


train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    collate_fn=custom_collate
)


In [30]:
features_batch, targets_batch = next(iter(train_loader))

print("Features batch shape:", features_batch.shape)
print("Targets batch shape:", targets_batch.shape)

Features batch shape: torch.Size([64, 2, 8])
Targets batch shape: torch.Size([64])
